 Grow a forest by following these steps:

Continuing the previous exercise, generate 1,000 subsets of the
training set, each containing 100 instances selected randomly. Hint:
you can use Scikit-Learn’s ShuffleSplit class for this.


b. Train one decision tree on each subset, using the best hyperparameter
values found in the previous exercise. Evaluate these 1,000 decision
trees on the test set. Since they were trained on smaller sets, these
decision trees will likely perform worse than the first decision tree,
achieving only about 80% accuracy.

In [4]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, ShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) Generate larger moons dataset
X, y = make_moons(n_samples=10_000, noise=0.4, random_state=42)

# 2) Train-test split (same style as q7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Build 1,000 random subsets of size 100 from training set
splitter = ShuffleSplit(n_splits=1000, train_size=100, random_state=42)

# 4) Best hyperparameters from q7
best_params = {
    "max_depth": None,
    "max_leaf_nodes": 30,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

trees = []
# 5) Train 1,000 trees and evaluate each on test set
acc_scores = []

for subset_idx, _ in splitter.split(X_train):
    X_subset = X_train[subset_idx]
    y_subset = y_train[subset_idx]

    tree = DecisionTreeClassifier(random_state=42, **best_params)
    tree.fit(X_subset, y_subset)
    trees.append(tree)

    y_pred = tree.predict(X_test)
    acc_scores.append(accuracy_score(y_test, y_pred))

acc_scores = np.array(acc_scores)

print("Number of trees:", len(acc_scores))
print("Average test accuracy:", acc_scores.mean())
print("Std of test accuracy:", acc_scores.std())
print("Min/Max test accuracy:", acc_scores.min(), acc_scores.max())

Number of trees: 1000
Average test accuracy: 0.79547
Std of test accuracy: 0.027012212053069626
Min/Max test accuracy: 0.668 0.8585


Now comes the magic. For each test set instance, generate the
predictions of the 1,000 decision trees, and keep only the most
frequent prediction (you can use SciPy’s mode() function for this).
This approach gives you majority-vote predictions over the test set.

In [5]:
from scipy.stats import mode
import numpy as np
from sklearn.metrics import accuracy_score
# 6) Collect all tree predictions on test set
# shape: (n_trees, n_test_samples)
all_tree_preds = np.array([tree.predict(X_test) for tree in trees])
# 7) Majority vote across trees (column-wise mode)
majority_vote_preds = mode(all_tree_preds, axis=0, keepdims=False).mode
# 8) Evaluate ensemble
ensemble_acc = accuracy_score(y_test, majority_vote_preds)
print("Majority-vote test accuracy:", ensemble_acc)

Majority-vote test accuracy: 0.863


This is a random forest!

Optimise random forest algos, Each split only considers a subset of features, not all. making them more robust.